# 합성 데이터 생성

실제 구매 로그가 없어 분석에 필요한 유저 데이터와 구매 이벤트 데이터를 합성으로 생성함.

- 유저 수 : 9,050명
- 이벤트 : view / cart / purchase 퍼널 구조

In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv('amazon_cleaned.csv')

print(df.shape)
print(df['product_id'].nunique())

(1465, 22)
1351


In [7]:
print(df['user_id'].head(3))

0    AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...
1    AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...
2    AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...
Name: user_id, dtype: object


In [8]:
user_list = (
    df['user_id']
    .str.split(',')   
    .explode()       
    .str.strip()     
    .unique()       
)

print(f"전체 유저 수: {len(user_list)}명")

전체 유저 수: 9050명


## 유저 데이터 생성

In [9]:
np.random.seed(42)  

date_range = pd.date_range('2023-01-01', '2023-12-31')

df_users = pd.DataFrame({
    'user_id'   : user_list,
    'join_date' : np.random.choice(date_range, size=len(user_list)),
    'user_grade': np.random.choice(['일반', 'VIP'], size=len(user_list), p=[0.8, 0.2])
})

print(df_users.shape)
df_users.head()

(9050, 3)


,user_id,join_date,user_grade
0,AG3D6O4STAQKAY2UVGEUV46KN35Q,2023-04-13,일반
1,AHMY5CWJMMK5BJRBBSNLYT3ONILA,2023-12-15,일반
2,AHCTC6ULH4XB6YHDY6PCH2R772LQ,2023-09-28,일반
3,AGYHHIERNXKA6P5T7CZLXKVPT7IQ,2023-04-17,일반
4,AG4OGOFWXJZTQ2HKYIOCOY3KXF2Q,2023-03-13,일반


## 구매 이벤트 만들기

In [10]:
np.random.seed(42)

product_list = df['product_id'].unique() 

events = []

for user in user_list:
    viewed = np.random.choice(product_list, 
                               size=np.random.randint(1, 6), 
                               replace=False)
    
    for product in viewed:
        event_date = np.random.choice(pd.date_range('2023-01-01', '2024-12-31'))
        events.append([user, product, event_date, 'view'])
        
        if np.random.random() < 0.3:
            events.append([user, product, event_date, 'cart'])
            
            if np.random.random() < 0.4:
                events.append([user, product, event_date, 'purchase'])

df_events = pd.DataFrame(events, columns=['user_id', 'product_id', 'event_date', 'event_type'])

print(df_events.shape)
df_events.head(10)

(38524, 4)


,user_id,product_id,event_date,event_type
0,AG3D6O4STAQKAY2UVGEUV46KN35Q,B07232M876,2023-03-22,view
1,AG3D6O4STAQKAY2UVGEUV46KN35Q,B074CWD7MS,2024-01-09,view
2,AG3D6O4STAQKAY2UVGEUV46KN35Q,B0B3G5XZN5,2023-07-19,view
3,AG3D6O4STAQKAY2UVGEUV46KN35Q,B0B3G5XZN5,2023-07-19,cart
4,AG3D6O4STAQKAY2UVGEUV46KN35Q,B017NC2IPM,2024-10-16,view
5,AHMY5CWJMMK5BJRBBSNLYT3ONILA,B09GP6FBZT,2024-02-03,view
6,AHMY5CWJMMK5BJRBBSNLYT3ONILA,B09GP6FBZT,2024-02-03,cart
7,AHCTC6ULH4XB6YHDY6PCH2R772LQ,B07NTKGW45,2023-02-28,view
8,AHCTC6ULH4XB6YHDY6PCH2R772LQ,B09P182Z2H,2023-02-19,view
9,AHCTC6ULH4XB6YHDY6PCH2R772LQ,B0116MIKKC,2023-01-19,view


In [11]:
funnel = df_events.groupby('event_type')['user_id'].nunique()
print(funnel)

event_type
cart        5551
purchase    2789
view        9050
Name: user_id, dtype: int64


In [12]:
view     = funnel['view']
cart     = funnel['cart']
purchase = funnel['purchase']

print(f"조회 → 장바구니 전환율 : {cart/view*100:.1f}%")
print(f"장바구니 → 구매 전환율 : {purchase/cart*100:.1f}%")
print(f"전체 전환율 (조회→구매): {purchase/view*100:.1f}%")

조회 → 장바구니 전환율 : 61.3%
장바구니 → 구매 전환율 : 50.2%
전체 전환율 (조회→구매): 30.8%


전체 전환율 30.8%는 합성 데이터 기준이며, 실제 서비스에서는 이보다 낮을 가능성이 있음.

In [14]:
df_events.to_csv('events.csv', index=False)
df_users.to_csv('users.csv', index=False)
print("저장 완료!")

저장 완료!


## 저장 완료

- events.csv: 구매 이벤트 로그
- users.csv: 유저 정보
